# Geospatial Data Analysis and Visualization
## Class 11 — Advanced Analysis & Geovisualization

**Keynote Speaker:** Md Wabidur Rahman, Researcher, Research and Integrated Thought (RIT)
**Class 11 · 23 October 2026**

---

### What this notebook covers

| Part | Topic | Main tool |
|------|------|---------|
| 1 | **The national trend** — a 25-year time series | `scipy.stats` |
| 2 | **Per-pixel trends** + significance | vectorized OLS |
| 3 | **Mann-Kendall and Sen's slope** (journal standard) | `pymannkendall` |
| 4 | **Trends by district** | zonal + trend |
| 5 | **Spatial autocorrelation** — Moran's I, LISA | `esda`, `libpysal` |
| 6 | **Hot-spot analysis** — Getis-Ord Gi* | `esda` |
| 7 | **Clustering** — climate zones | `scikit-learn` |
| 8 | **Network analysis** — connectivity and routes | `networkx` |
| 9 | **Interactive maps** | `folium` |
| 10 | **Journal-ready figures and tables** | `matplotlib` |

---

### Today's final goal

By the end of today you will hold the complete **Results** section of a research paper —

- **Figure 1** — the study area and the mean climate
- **Figure 2** — maps of the warming trend (with significance)
- **Figure 3** — spatial clusters and climate zones
- **Tables 1–3** — as CSV, ready to drop into a manuscript

All at **300 dpi**, in TIFF/PDF/PNG — which is what almost every journal asks for.

> **The research question:** *How did near-surface air temperature and relative humidity
> change over Bangladesh between 2000 and 2024, and what is the spatial structure of that change?*

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  SETUP — locating and preparing the data  (identical in every notebook)
# ════════════════════════════════════════════════════════════════════
# This cell finds your four datasets by itself:
#   * monthly_mean_data_*.zip / data_0.nc   (ERA5 monthly)
#   * hourly_mean_data_*.zip  / data_0.nc   (ERA5 hourly)
#   * BGD_adm.zip                           (GADM shapefile)
#   * bgd_adm_bbs_*_shp.zip                 (BBS/HDX shapefile)
# If it finds the zips it extracts them into ./data/. If they are already extracted it uses them as they are.

from pathlib import Path
import zipfile, sys, os

DATA = Path.cwd() / "data"
OUT  = Path.cwd() / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# Where to look
SEARCH_ROOTS = [Path.cwd(), Path.cwd()/"data", Path.cwd().parent,
                Path.home()/"Downloads", Path.home()/"Desktop", Path.home()]

def _find(patterns, roots=SEARCH_ROOTS, maxdepth=3):
    """Return the first file that matches the pattern."""
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            try:
                hits = sorted(root.glob(pat)) + sorted(root.glob("*/" + pat))
                if maxdepth >= 3:
                    hits += sorted(root.glob("*/*/" + pat))
                for h in hits:
                    if h.exists():
                        return h
            except Exception:
                pass
    return None

def _ensure(zip_patterns, member_pattern, dest):
    """If member_pattern is not in dest, find the zip and extract it there."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    got = sorted(dest.rglob(member_pattern))
    if got:
        return got[0]
    z = _find(zip_patterns)
    if z is not None:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        got = sorted(dest.rglob(member_pattern))
        if got:
            print(f"  extracted {z.name}  ->  {dest.name}/")
            return got[0]
    return None

print("Looking for the data ...")
NC_MONTHLY = _ensure(["*monthly_mean*zip"], "*.nc", DATA/"nc_monthly")
NC_HOURLY  = _ensure(["*hourly_mean*zip"],  "*.nc", DATA/"nc_hourly")
SHP_GADM   = _ensure(["*BGD_adm*.zip"], "BGD_adm2.shp", DATA/"shp_adm")
SHP_BBS    = _ensure(["*bgd_adm_bbs*shp*.zip", "*bgd_adm_bbs*.zip"],
                     "bgd_admbnda_adm2*.shp", DATA/"shp_bbs")

# GADM is optional -- only the GADM versus BBS comparison in Class 08 needs
# it (see the README).
status = {"ERA5 monthly (.nc)": NC_MONTHLY, "ERA5 hourly (.nc)": NC_HOURLY,
          "BBS/HDX shapefile": SHP_BBS}
print()
ok = True
for k, v in status.items():
    print(f"  {'✓' if v else '✗'}  {k:24s} {v if v else 'not found'}")
    ok = ok and (v is not None)
print(f"  {'✓' if SHP_GADM else '·'}  {'GADM (optional)':24s} "
      f"{SHP_GADM if SHP_GADM else 'absent -- only the Class 08 comparison is skipped'}")
print()
if ok:
    print("All required data ready")
else:
    print("Some required files were not found — put the zip files next to this notebook and run again.")

In [ ]:
# -- Libraries and shared styling (identical in every notebook) ------------
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import geopandas as gpd, xarray as xr

# Publication-ready defaults so the figures can go straight into a journal
plt.rcParams.update({
    "figure.dpi": 110,          # for viewing on screen
    "savefig.dpi": 300,         # for saving to a file (journals usually want >=300)
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.autolayout": False,
})

# Brand colours — the same in every notebook
C_WARM, C_COOL, C_ACC, C_INK = "#B4562F", "#3D7EA6", "#E0A458", "#1B3A5C"

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("geopandas   ", gpd.__version__)
print("xarray      ", xr.__version__)
print("matplotlib  ", matplotlib.__version__)

In [ ]:
from scipy import stats
import pymannkendall as mk

BBSDIR = SHP_BBS.parent
country  = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm0_*.shp"))[0])[["geometry"]]
division = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm1_*.shp"))[0])[["ADM1_EN", "geometry"]]
district = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm2_*.shp"))[0])[
    ["ADM2_EN", "ADM1_EN", "geometry"]]

ds = xr.open_dataset(NC_MONTHLY).squeeze(drop=True)
ds["t_c"] = ds["t"] - 273.15
dsh = xr.open_dataset(NC_HOURLY).squeeze(drop=True)
dsh["t_c"] = dsh["t"] - 273.15

print(f"districts {len(district)} | divisions {len(division)}")
print(f"monthly ERA5 : {dict(ds.sizes)}")
print(f"hourly       : {dict(dsh.sizes)}")

---
# Part 1 — The national trend (national time series)

## 1.1 Why an area-weighted mean?

Every cell of the grid covers the same number of **degrees**, but not the same **area** —
the cells are narrower further north. So a plain mean gives the northern cells too much weight.

The fix: weight by **cos(latitude)** —

$$\bar{T} = \frac{\sum_i T_i \cos(\phi_i)}{\sum_i \cos(\phi_i)}$$

This is the **standard practice** in climate research. Skip it and a reviewer will notice.

In [ ]:
# the area-weighted national mean
w = np.cos(np.deg2rad(ds.latitude))
t_sp = ds.t_c.weighted(w).mean(dim=["latitude", "longitude"])
r_sp = ds.r.weighted(w).mean(dim=["latitude", "longitude"])

# compared with the plain mean
t_plain = ds.t_c.mean(dim=["latitude", "longitude"])
print(f"area-weighted mean : {float(t_sp.mean()):.4f} C")
print(f"plain mean         : {float(t_plain.mean()):.4f} C")
print(f"difference         : {abs(float(t_sp.mean())-float(t_plain.mean())):.4f} C")
print("-> the difference is small here (the area is small), but the rule must be followed.")

# the annual means
t_ann = t_sp.groupby("valid_time.year").mean()
r_ann = r_sp.groupby("valid_time.year").mean()
years = t_ann.year.values.astype(float)

print(f"\nAnnual mean temperature ({int(years[0])}-{int(years[-1])}):")
print(f"  mean of the first 5 years : {float(t_ann[:5].mean()):.3f} C")
print(f"  mean of the last 5 years  : {float(t_ann[-5:].mean()):.3f} C")
print(f"  difference                : {float(t_ann[-5:].mean())-float(t_ann[:5].mean()):+.3f} C")

### Why three methods?

| Method | Kind | Advantage | Limitation |
|--------|------|---------|-------------|
| **OLS (a straight line)** | parametric | simple, gives R² | affected by outliers |
| **Mann-Kendall** | non-parametric | no assumption about the distribution | does not give the **rate** |
| **Sen's slope** | non-parametric | an outlier-resistant **rate** | slow on large data |

> **The usual journal convention:** climate research uses **Mann-Kendall (for the significance)
> plus Sen's slope (for the rate)** together. Showing OLS as well is good practice,
> because when all three agree the result is far more convincing.

In [ ]:
# the national trend — by three methods
res_t = stats.linregress(years, t_ann.values)
res_r = stats.linregress(years, r_ann.values)
mk_t = mk.original_test(t_ann.values)
mk_r = mk.original_test(r_ann.values)

print("═" * 62)
print("Temperature trend, 2000-2024")
print("═" * 62)
print(f"  OLS linear trend : {res_t.slope*10:+.3f} C/decade   "
      f"(p = {res_t.pvalue:.4f}, R² = {res_t.rvalue**2:.3f})")
print(f"  Mann-Kendall     : {mk_t.trend}  (p = {mk_t.p:.4f}, tau = {mk_t.Tau:.3f})")
print(f"  Sen's slope      : {mk_t.slope*10:+.3f} C/decade")
print(f"  total warming (over 25 years): {res_t.slope*25:+.2f} C")
print()
print("═" * 62)
print("Relative humidity trend")
print("═" * 62)
print(f"  OLS linear trend : {res_r.slope*10:+.3f} %/decade   (p = {res_r.pvalue:.4f})")
print(f"  Mann-Kendall     : {mk_r.trend}  (p = {mk_r.p:.4f})")
print()
print("Interpretation: temperature is rising significantly (p < 0.05),")
print("                but the change in humidity is not statistically significant.")

In [ ]:
# the national time-series figure
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# (a) temperature
ax[0].plot(years, t_ann.values, "o-", color=C_WARM, lw=1.6, ms=5,
           label="Annual mean", zorder=3)
ax[0].plot(years, res_t.intercept + res_t.slope*years, "--", color=C_INK, lw=2,
           label=f"OLS: {res_t.slope*10:+.3f} °C/decade (p={res_t.pvalue:.3f})")
roll = pd.Series(t_ann.values, index=years).rolling(5, center=True).mean()
ax[0].plot(years, roll.values, "-", color="k", lw=1.2, alpha=0.6,
           label="5-year running mean")
ax[0].axhline(float(t_ann[:10].mean()), color="gray", ls=":", lw=1,
              label="2000–2009 baseline")
ax[0].set_ylabel("Temperature (°C)")
ax[0].set_title("(a) Area-weighted annual mean air temperature over Bangladesh, ERA5 1000 hPa")
ax[0].legend(fontsize=8.5, loc="upper left")

# (b) humidity
ax[1].plot(years, r_ann.values, "s-", color=C_COOL, lw=1.6, ms=4.5,
           label="Annual mean")
ax[1].plot(years, res_r.intercept + res_r.slope*years, "--", color=C_INK, lw=2,
           label=f"OLS: {res_r.slope*10:+.3f} %/decade (p={res_r.pvalue:.3f}, n.s.)")
ax[1].set_ylabel("Relative humidity (%)"); ax[1].set_xlabel("Year")
ax[1].set_title("(b) Area-weighted annual mean relative humidity")
ax[1].legend(fontsize=8.5, loc="upper left")

plt.tight_layout()
plt.savefig(OUT/"fig_nb4_national_timeseries.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# the seasonal and diurnal cycle — using the hourly file
wh = np.cos(np.deg2rad(dsh.latitude))
h_sp = dsh.t_c.weighted(wh).mean(dim=["latitude", "longitude"])
dfh = h_sp.to_dataframe(name="t").reset_index()
dfh["hour_utc"] = dfh["valid_time"].dt.hour
dfh["hour_bd"] = (dfh["hour_utc"] + 6) % 24          # Bangladesh = UTC+6
dfh["month"] = dfh["valid_time"].dt.month
season = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJAS", 7: "JJAS", 8: "JJAS", 9: "JJAS", 10: "ON", 11: "ON"}
dfh["season"] = dfh["month"].map(season)

diurnal = dfh.groupby("hour_bd")["t"].mean()
clim = ds.t_c.weighted(w).mean(dim=["latitude", "longitude"]).groupby("valid_time.month").mean()

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax[0].plot(clim.month, clim.values, "o-", color=C_WARM, lw=2, ms=6)
ax[0].set_xticks(range(1, 13))
ax[0].set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax[0].set_xlabel("Month"); ax[0].set_ylabel("Temperature (°C)")
ax[0].set_title("(a) Mean annual cycle, 2000–2024")

for s, col in zip(["DJF", "MAM", "JJAS", "ON"], [C_COOL, C_ACC, C_WARM, "#7FB069"]):
    g = dfh[dfh["season"] == s].groupby("hour_bd")["t"].mean().sort_index()
    ax[1].plot(g.index, g.values, "o-", lw=1.8, ms=4, label=s, color=col)
ax[1].set_xlabel("Hour of day (Bangladesh Standard Time, UTC+6)")
ax[1].set_ylabel("Temperature (°C)")
ax[1].set_title("(b) Diurnal cycle by season (ERA5 hourly product)")
ax[1].legend(fontsize=8.5, ncol=4)
plt.tight_layout()
plt.savefig(OUT/"fig_nb4_cycles.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"warmest month : number {int(clim.idxmax())} ({float(clim.max()):.2f} C)")
print(f"coolest month : number {int(clim.idxmin())} ({float(clim.min()):.2f} C)")
print(f"annual range  : {float(clim.max())-float(clim.min()):.2f} C")
print(f"\ndaily maximum (within the available hours): {int(diurnal.idxmax())}:00 BST "
      f"({diurnal.max():.2f} °C)")
print("Note: this product covers only UTC 07-18 (BST 13-24), not the full 24 hours.")

---
# Part 2 — Per-pixel trends (the trend map)

A national mean gives one number. But the real research question is —
**"is the warming the same everywhere?"**

To answer that you have to compute a trend **separately in every grid cell**.
You could loop over the 841 cells, but the **vectorized** approach is a thousand times faster.

In [ ]:
# the cube of annual means (year x lat x lon)
annual = ds.t_c.groupby("valid_time.year").mean("valid_time")
Y = annual.values
yr = annual.year.values.astype(float)
nt = len(yr)

# -- Vectorized OLS (every pixel at once) ---------------------------
ym, Ym = yr.mean(), Y.mean(axis=0)
Sxy = ((yr - ym)[:, None, None] * (Y - Ym)).sum(axis=0)
Sxx = ((yr - ym) ** 2).sum()

slope = Sxy / Sxx                       # °C per year
intercept = Ym - slope * ym
resid = Y - (intercept + slope * yr[:, None, None])
se = np.sqrt((resid ** 2).sum(axis=0) / (nt - 2) / Sxx)
tval = slope / se
pval = 2 * (1 - stats.t.cdf(np.abs(tval), nt - 2))

trend_dec = slope * 10                  # °C per decade

print(f"trend (C/decade): minimum {trend_dec.min():+.3f}  "
      f"maximum {trend_dec.max():+.3f}  mean {trend_dec.mean():+.3f}")
print(f"significant (p < 0.05): {(pval < 0.05).sum()} / {pval.size} cells "
      f"({100*(pval < 0.05).mean():.1f}%)")

# a check — verifying one cell against scipy
i, j = 14, 14
chk = stats.linregress(yr, Y[:, i, j])
print(f"\ncheck cell ({i},{j}): vectorized {trend_dec[i,j]:.4f}  vs  "
      f"scipy {chk.slope*10:.4f}   p: {pval[i,j]:.4f} vs {chk.pvalue:.4f}  ✓")

In [ ]:
# turning the trend into a DataArray and clipping it to Bangladesh
import rioxarray
def to_raster(arr, name):
    da = xr.DataArray(arr, coords={"latitude": ds.latitude, "longitude": ds.longitude},
                      dims=("latitude", "longitude"), name=name)
    return (da.rio.write_crs("EPSG:4326")
            .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude"))

trend_da = to_raster(trend_dec, "trend")
pval_da = to_raster(pval, "pval")

trend_bd = trend_da.rio.clip(country.geometry, country.crs, all_touched=True, drop=True)
pval_bd = pval_da.rio.clip(country.geometry, country.crs, all_touched=True, drop=True)

print(f"Trend inside Bangladesh:")
print(f"  mean        : {float(trend_bd.mean()):+.3f} C/decade")
print(f"  minimum     : {float(trend_bd.min()):+.3f}")
print(f"  maximum     : {float(trend_bd.max()):+.3f}")
sig_frac = float((pval_bd < 0.05).sum()) / float(pval_bd.notnull().sum())
print(f"  significant : {100*sig_frac:.1f}% of cells (p < 0.05)")

In [ ]:
# the trend map — with significance stippling (the journal convention)
fig, ax = plt.subplots(1, 2, figsize=(13.5, 6))

# (a) the trend
vmax = float(np.nanmax(np.abs(trend_bd)))
im = trend_bd.plot(ax=ax[0], cmap="RdYlBu_r", vmin=0, vmax=vmax, add_colorbar=False)
cb = plt.colorbar(im, ax=ax[0], shrink=0.78, pad=0.02)
cb.set_label("Temperature trend (°C decade⁻¹)")

# dots on the significant cells
LO, LA = np.meshgrid(pval_bd.longitude.values, pval_bd.latitude.values)
sig_mask = (pval_bd.values < 0.05) & np.isfinite(trend_bd.values)
ax[0].scatter(LO[sig_mask], LA[sig_mask], s=3.5, c="k", alpha=0.65, marker=".", linewidths=0)
division.boundary.plot(ax=ax[0], color="k", linewidth=0.5)
country.boundary.plot(ax=ax[0], color="k", linewidth=1.4)
ax[0].set_title("(a) Warming trend 2000–2024\ndots = significant at p < 0.05")

# (b) the p-value
im2 = pval_bd.plot(ax=ax[1], cmap="viridis_r", vmin=0, vmax=0.2, add_colorbar=False)
cb2 = plt.colorbar(im2, ax=ax[1], shrink=0.78, pad=0.02)
cb2.set_label("p-value")
country.boundary.plot(ax=ax[1], color="w", linewidth=1.4)
ax[1].set_title("(b) Statistical significance (p-value)")

for a in ax:
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout()
plt.savefig(OUT/"fig_nb4_trend_map.png", dpi=300, bbox_inches="tight")
plt.show()

> **Why stippling?** In a climate map, showing the trend alone is not enough —
> you also have to show where the trend is **statistically significant**.
> Almost all major studies, the IPCC reports included, use dots (stippling) or hatching
> to convey this. Reviewers look for it.

---
# Part 3 — Mann-Kendall and Sen's Slope

OLS assumes the residuals are normally distributed. With climate data that is
not always true. So a **non-parametric** method is safer —

- **the Mann-Kendall test** — is there a trend at all (yes/no + p)
- **the Theil-Sen estimator** — the rate of the trend, resistant to outliers

In [ ]:
# MK + Sen in every cell (841 cells — under a second)
import time
t0 = time.time()
sen = np.full(Y.shape[1:], np.nan)
mkp = np.full(Y.shape[1:], np.nan)
for i in range(Y.shape[1]):
    for j in range(Y.shape[2]):
        r = mk.original_test(Y[:, i, j])
        sen[i, j] = r.slope * 10        # C/decade
        mkp[i, j] = r.p
print(f"MK + Sen computation time: {time.time()-t0:.2f} seconds ({Y[0].size} cells)")

sen_da = to_raster(sen, "sen").rio.clip(country.geometry, country.crs,
                                        all_touched=True, drop=True)
mkp_da = to_raster(mkp, "mkp").rio.clip(country.geometry, country.crs,
                                        all_touched=True, drop=True)

print(f"\nSen's slope (Bangladesh): mean {float(sen_da.mean()):+.3f} C/decade")
print(f"MK significant          : "
      f"{100*float((mkp_da<0.05).sum())/float(mkp_da.notnull().sum()):.1f}% of cells")

# agreement between the two methods
a_, b_ = trend_bd.values.ravel(), sen_da.values.ravel()
ok = np.isfinite(a_) & np.isfinite(b_)
print(f"\nCorrelation of OLS versus Sen: r = {np.corrcoef(a_[ok], b_[ok])[0,1]:.4f}")
print("-> very high agreement, so the result does not depend on the method (it is robust).")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))

im0 = trend_bd.plot(ax=ax[0], cmap="RdYlBu_r", vmin=0, vmax=0.45, add_colorbar=False)
plt.colorbar(im0, ax=ax[0], shrink=0.75, label="°C decade⁻¹")
ax[0].set_title("(a) OLS linear trend")

im1 = sen_da.plot(ax=ax[1], cmap="RdYlBu_r", vmin=0, vmax=0.45, add_colorbar=False)
plt.colorbar(im1, ax=ax[1], shrink=0.75, label="°C decade⁻¹")
ax[1].set_title("(b) Theil–Sen slope")

ax[2].scatter(a_[ok], b_[ok], s=12, alpha=0.5, color=C_COOL, edgecolor="none")
lims = [min(a_[ok].min(), b_[ok].min()), max(a_[ok].max(), b_[ok].max())]
ax[2].plot(lims, lims, "--", color=C_WARM, lw=1.6, label="1:1 line")
ax[2].set_xlabel("OLS trend (°C decade⁻¹)")
ax[2].set_ylabel("Sen's slope (°C decade⁻¹)")
ax[2].set_title(f"(c) Agreement (r = {np.corrcoef(a_[ok], b_[ok])[0,1]:.3f})")
ax[2].legend(fontsize=8.5)

for a in ax[:2]:
    country.boundary.plot(ax=a, color="k", linewidth=1.2)
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("")
plt.tight_layout(); plt.show()

---
# Part 4 — Trends by district

Policymakers do not think in grid cells — they think in **districts**. So translating
results into administrative units is an important step in research.

In [ ]:
# the zonal mean for each district (the all_touched method of Notebook 3)
t_mean_da = to_raster(ds.t_c.mean("valid_time").values, "t")
rh_mean_da = to_raster(ds.r.mean("valid_time").values, "rh")

recs = []
for _, row in district.iterrows():
    g = [row.geometry]
    try:
        recs.append({
            "ADM2_EN": row["ADM2_EN"], "ADM1_EN": row["ADM1_EN"],
            "trend": float(trend_da.rio.clip(g, district.crs, all_touched=True, drop=True).mean()),
            "sen": float(to_raster(sen, "s").rio.clip(g, district.crs, all_touched=True, drop=True).mean()),
            "pval": float(pval_da.rio.clip(g, district.crs, all_touched=True, drop=True).mean()),
            "t_mean": float(t_mean_da.rio.clip(g, district.crs, all_touched=True, drop=True).mean()),
            "rh_mean": float(rh_mean_da.rio.clip(g, district.crs, all_touched=True, drop=True).mean()),
        })
    except Exception:
        recs.append({"ADM2_EN": row["ADM2_EN"], "ADM1_EN": row["ADM1_EN"],
                     "trend": np.nan, "sen": np.nan, "pval": np.nan,
                     "t_mean": np.nan, "rh_mean": np.nan})

dz = district.merge(pd.DataFrame(recs), on=["ADM2_EN", "ADM1_EN"], how="left")
print(f"districts: {len(dz)} | empty: {int(dz['trend'].isna().sum())}")
print(f"range of the trend: {dz['trend'].min():+.3f} -> {dz['trend'].max():+.3f} C/decade\n")

print("The eight fastest-warming districts:")
print(dz.nlargest(8, "trend")[["ADM2_EN", "ADM1_EN", "trend", "pval"]]
      .round(3).to_string(index=False))
print("\nThe eight slowest-warming districts:")
print(dz.nsmallest(8, "trend")[["ADM2_EN", "ADM1_EN", "trend", "pval"]]
      .round(3).to_string(index=False))

In [ ]:
# the summary by division
div_sum = (dz.groupby("ADM1_EN")
           .agg(n=("ADM2_EN", "size"), trend=("trend", "mean"),
                t_mean=("t_mean", "mean"), rh_mean=("rh_mean", "mean"))
           .round(3).sort_values("trend", ascending=False))
print("Warming by division (C/decade):")
print(div_sum.to_string())

fig, ax = plt.subplots(figsize=(9, 4.2))
d2 = div_sum.sort_values("trend")
ax.barh(d2.index, d2["trend"], color=[C_WARM if v > div_sum["trend"].mean()
                                      else C_COOL for v in d2["trend"]],
        edgecolor="k", linewidth=0.5)
ax.axvline(float(dz["trend"].mean()), color="k", ls="--", lw=1.2,
           label=f"National mean = {dz['trend'].mean():.3f}")
ax.set_xlabel("Warming trend (°C decade⁻¹)")
ax.set_title("Warming rate by division, 2000–2024")
ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

---
# Part 5 — Spatial autocorrelation

## 5.1 Tobler's first law

> *"Everything is related to everything else, but near things are more related than distant things."*

The question: **are the fast-warming districts clustered together, or scattered at random?**

**Moran's I** answers it —

| Value of I | Meaning |
|----------|------|
| **close to +1** | strong clustering (nearby districts have similar values) |
| **close to 0** | random (no spatial structure) |
| **close to −1** | like a chessboard (nearby values are opposite) |

## 5.2 Spatial weights — who counts as a "neighbour"?

| Method | Definition |
|--------|---------|
| **Queen** | a neighbour if it touches along an edge or at a corner |
| **Rook** | only if it touches along an edge |
| **KNN** | the k nearest ones |

In [ ]:
from libpysal.weights import Queen, KNN
import esda

dz_ok = dz.dropna(subset=["trend"]).reset_index(drop=True)

wq = Queen.from_dataframe(dz_ok, use_index=False)
wq.transform = "r"                      # row standardisation
print(f"Queen contiguity: {wq.n} units, mean neighbours {wq.mean_neighbors:.2f}")
if len(wq.islands):
    print(f"  Disconnected (island) units: {len(wq.islands)} -> solved with KNN")
    wq = KNN.from_dataframe(dz_ok, k=5); wq.transform = "r"
    print(f"  KNN(k=5) is being used")

mi = esda.Moran(dz_ok["trend"].values, wq)
print(f"\n=== Global Moran's I — the warming trend ===")
print(f"  I          = {mi.I:.4f}")
print(f"  expected E[I] = {mi.EI:.4f}  (under randomness)")
print(f"  z-score    = {mi.z_sim:.3f}")
print(f"  p (simulated) = {mi.p_sim:.4f}")
print(f"\n  -> {'strong positive spatial clustering' if mi.I > 0.3 and mi.p_sim < 0.05 else 'weak / not significant'}")

mi_t = esda.Moran(dz_ok["t_mean"].values, wq)
print(f"\nFor comparison — Moran's I of the mean temperature = {mi_t.I:.4f} (p = {mi_t.p_sim:.4f})")

In [ ]:
# the Moran scatterplot — understanding the axes matters
from libpysal.weights import lag_spatial
z_std = (dz_ok["trend"] - dz_ok["trend"].mean()) / dz_ok["trend"].std()
z_lag = lag_spatial(wq, z_std.values)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(z_std, z_lag, s=42, color=C_COOL, edgecolor="k", linewidth=0.4, alpha=0.8)
mor_b, mor_a = np.polyfit(z_std, z_lag, 1)
mor_xs = np.linspace(z_std.min(), z_std.max(), 50)
ax[0].plot(mor_xs, mor_a + mor_b*mor_xs, "-", color=C_WARM, lw=2,
           label=f"slope = Moran's I = {mi.I:.3f}")
ax[0].axhline(0, color="k", lw=0.8); ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_xlabel("Standardised warming trend (z)")
ax[0].set_ylabel("Spatial lag of z (neighbours' mean)")
ax[0].set_title("Moran scatterplot")
ax[0].legend(fontsize=9)
for xx, yy, lab in [(1.4, 1.4, "HH"), (-1.4, 1.4, "LH"),
                    (-1.4, -1.4, "LL"), (1.4, -1.4, "HL")]:
    ax[0].text(xx, yy, lab, fontsize=13, fontweight="bold",
               color="gray", ha="center", va="center")

# simulation-based significance
ax[1].hist(mi.sim, bins=40, color="lightgray", edgecolor="white")
ax[1].axvline(mi.I, color=C_WARM, lw=2.5, label=f"Observed I = {mi.I:.3f}")
ax[1].axvline(mi.EI, color="k", ls="--", lw=1.5, label=f"E[I] = {mi.EI:.3f}")
ax[1].set_xlabel("Moran's I under spatial randomness")
ax[1].set_ylabel("Frequency")
ax[1].set_title(f"Reference distribution (999 permutations), p = {mi.p_sim:.3f}")
ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

### 5.3 LISA — where are the local clusters?

Global Moran's I gives one number ("there is clustering"). But it does not say **where**.
**LISA** (*Local Indicators of Spatial Association*) gives a separate value for each
district —

| Class | Meaning |
|--------|------|
| **HH** (High-High) | a **warming hot-spot** — high itself, with high neighbours |
| **LL** (Low-Low) | a **cold-spot** — low itself, with low neighbours |
| **HL** (High-Low) | a **spatial outlier** — high itself, with low neighbours |
| **LH** (Low-High) | low itself, with high neighbours |

In [ ]:
lisa = esda.Moran_Local(dz_ok["trend"].values, wq, seed=42)
sig = lisa.p_sim < 0.05
labels = np.array(["Not significant", "HH (hot-spot)", "LH", "LL (cold-spot)", "HL"])
cat = np.where(sig, lisa.q, 0)
dz_ok["lisa"] = labels[cat]

print("LISA classification:")
for k, v in dz_ok["lisa"].value_counts().items():
    print(f"  {k:22s} {v:3d} districts")

print("\nWarming hot-spots (HH):")
hh = dz_ok[dz_ok["lisa"] == "HH (hot-spot)"]
print("  ", ", ".join(sorted(hh["ADM2_EN"])) if len(hh) else "none")
print("\n❄ Cold-spot (LL):")
ll = dz_ok[dz_ok["lisa"] == "LL (cold-spot)"]
print("  ", ", ".join(sorted(ll["ADM2_EN"])) if len(ll) else "none")

In [ ]:
# the LISA cluster map
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

cmap_d = {"Not significant": "#e8e8e8", "HH (hot-spot)": "#c0392b",
          "LL (cold-spot)": "#2874a6", "LH": "#a9cce3", "HL": "#f5b7b1"}

fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.2))

dz_ok.plot(column="trend", ax=ax[0], cmap="RdYlBu_r", legend=True,
           edgecolor="white", linewidth=0.3,
           legend_kwds={"label": "Trend (°C decade⁻¹)", "shrink": 0.75})
ax[0].set_title("(a) District warming trend")

dz_ok["color"] = dz_ok["lisa"].map(cmap_d)
dz_ok.plot(ax=ax[1], color=dz_ok["color"], edgecolor="white", linewidth=0.35)
ax[1].set_title(f"(b) LISA clusters (Moran's I = {mi.I:.3f}, p = {mi.p_sim:.3f})")
ax[1].legend(handles=[mpatches.Patch(color=c, label=k) for k, c in cmap_d.items()],
             loc="upper right", fontsize=7.5, frameon=True, framealpha=0.92)

for a in ax:
    division.boundary.plot(ax=a, color="k", linewidth=0.5)
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout()
plt.savefig(OUT/"fig_nb4_lisa.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Part 6 — Hot-spot analysis: Getis-Ord Gi*

LISA and Gi* both look for "clusters", but in slightly different ways —

| | **LISA (Local Moran)** | **Getis-Ord Gi\*** |
|---|---|---|
| What it measures | whether a value **matches** its neighbours | whether the surrounding values are **high or low** |
| Detects HL/LH? | yes (it also shows outliers) | no |
| Output | four classes | a z-score (+ = hot, − = cold) |
| When | understanding structure and outliers | building a **hot-spot map** |

In [ ]:
g = esda.G_Local(dz_ok["trend"].values, wq, seed=42)
dz_ok["gi_z"] = g.Zs
dz_ok["gi_p"] = g.p_sim

def gi_class(z, p):
    if p >= 0.05:  return "Not significant"
    if z >= 2.58:  return "Hot-spot 99%"
    if z >= 1.96:  return "Hot-spot 95%"
    if z <= -2.58: return "Cold-spot 99%"
    if z <= -1.96: return "Cold-spot 95%"
    return "Not significant"

dz_ok["gi_class"] = [gi_class(z, p) for z, p in zip(g.Zs, g.p_sim)]
print("Getis-Ord Gi* classification:")
for k, v in dz_ok["gi_class"].value_counts().items():
    print(f"  {k:20s} {v:3d}")
print(f"\nz-score range: {g.Zs.min():.2f} -> {g.Zs.max():.2f}")

gi_cols = {"Hot-spot 99%": "#99000d", "Hot-spot 95%": "#fb6a4a",
           "Not significant": "#f0f0f0",
           "Cold-spot 95%": "#6baed6", "Cold-spot 99%": "#08519c"}

fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.2))
dz_ok.plot(column="gi_z", ax=ax[0], cmap="RdBu_r", legend=True,
           edgecolor="white", linewidth=0.3, vmin=-3, vmax=3,
           legend_kwds={"label": "Gi* z-score", "shrink": 0.75})
ax[0].set_title("(a) Getis-Ord Gi* z-scores")

dz_ok.plot(ax=ax[1], color=dz_ok["gi_class"].map(gi_cols),
           edgecolor="white", linewidth=0.35)
ax[1].set_title("(b) Hot-spot / cold-spot classification")
ax[1].legend(handles=[mpatches.Patch(color=c, label=k) for k, c in gi_cols.items()],
             loc="upper right", fontsize=7.5, frameon=True, framealpha=0.92)
for a in ax:
    division.boundary.plot(ax=a, color="k", linewidth=0.5)
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("")
plt.tight_layout(); plt.show()

---
# Part 7 — Clustering: building climate zones

**K-Means** divides the data into k clusters so that the members of a cluster are
close to one another. We will use three features —
the mean temperature, the mean humidity, and the warming rate.

> **⚠ Standardisation is mandatory:** temperature is ~25 (°C), humidity ~75 (%),
> the trend ~0.2 (°C/decade). Without matching the scales, humidity would dominate the result.
> So `StandardScaler` comes first — giving every feature a mean of 0 and an SD of 1.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

feat = ["t_mean", "rh_mean", "trend"]
X = dz_ok[feat].values
Xs = StandardScaler().fit_transform(X)

# how many clusters? — elbow and silhouette
ks, inertia, sil = range(2, 9), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs, km.labels_))

best_k = list(ks)[int(np.argmax(sil))]
print("Choosing k:")
for k, i_, s_ in zip(ks, inertia, sil):
    print(f"  k={k}: inertia {i_:7.2f}   silhouette {s_:.4f}"
          + ("   <- highest" if k == best_k else ""))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ks, inertia, "o-", color=C_COOL, lw=2)
ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia"); ax[0].set_title("Elbow method")
ax[1].plot(ks, sil, "o-", color=C_WARM, lw=2)
ax[1].axvline(best_k, color="k", ls="--", lw=1.2)
ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette"); ax[1].set_title("Silhouette score")
plt.tight_layout(); plt.show()

In [ ]:
K = 4                                    # four zones, for interpretability
km = KMeans(n_clusters=K, n_init=25, random_state=42).fit(Xs)
dz_ok["zone"] = km.labels_

prof = dz_ok.groupby("zone")[feat].mean().round(3)
prof["n"] = dz_ok.groupby("zone").size()
print("Zone characteristics (cluster profile):")
print(prof.to_string())

# giving meaningful names — ordered by temperature
order = prof.sort_values("t_mean").index.tolist()
names = {order[0]: "Cool north", order[1]: "Moderate central",
         order[2]: "Warm humid", order[3]: "Warm west"}
dz_ok["zone_name"] = dz_ok["zone"].map(names)

print("\nDistricts by zone:")
for zi in order:
    members = sorted(dz_ok[dz_ok["zone"] == zi]["ADM2_EN"].tolist())
    print(f"\n  {names[zi]} (n={len(members)}, mean T {prof.loc[zi,'t_mean']:.2f} C, "
          f"trend {prof.loc[zi,'trend']:+.3f}):")
    print("    " + ", ".join(members))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.2))

zone_cols = {"Cool north": "#4575b4", "Moderate central": "#91bfdb",
             "Warm humid": "#fc8d59", "Warm west": "#d73027"}
dz_ok.plot(ax=ax[0], color=dz_ok["zone_name"].map(zone_cols),
           edgecolor="white", linewidth=0.35)
division.boundary.plot(ax=ax[0], color="k", linewidth=0.5)
ax[0].set_title(f"(a) Climate zones from K-Means (k={K})")
ax[0].legend(handles=[mpatches.Patch(color=c, label=k) for k, c in zone_cols.items()],
             loc="upper right", fontsize=7.5, frameon=True, framealpha=0.92)
ax[0].set_xlabel("Longitude (°E)"); ax[0].set_ylabel("Latitude (°N)")

for zname, c in zone_cols.items():
    s = dz_ok[dz_ok["zone_name"] == zname]
    ax[1].scatter(s["t_mean"], s["trend"], s=s["rh_mean"]*1.6, color=c,
                  edgecolor="k", linewidth=0.4, alpha=0.85, label=zname)
ax[1].set_xlabel("Mean temperature (°C)")
ax[1].set_ylabel("Warming trend (°C decade⁻¹)")
ax[1].set_title("(b) Feature space (bubble size ∝ relative humidity)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT/"fig_nb4_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Part 8 — Network analysis

A spatial network lets you analyse connectivity, distance and accessibility.
Here we build a network from the district centroids and find the **shortest path**.

> **A real road network:** to take one from OpenStreetMap, use `osmnx` —
> ```python
> import osmnx as ox
> G = ox.graph_from_place("Dhaka, Bangladesh", network_type="drive")
> route = ox.shortest_path(G, orig_node, dest_node, weight="length")
> ```
> That needs internet, so here we show a centroid-based network instead —
> the idea and the code are the same.

In [ ]:
import networkx as nx
from scipy.spatial import Delaunay

dm = dz_ok.to_crs(3106)
cent = dm.geometry.centroid
pts = np.c_[cent.x.values, cent.y.values]
names_d = dz_ok["ADM2_EN"].tolist()

# Delaunay triangulation -> neighbour connections
tri = Delaunay(pts)
G = nx.Graph()
for i, nm in enumerate(names_d):
    G.add_node(i, name=nm, x=pts[i, 0], y=pts[i, 1])
for s in tri.simplices:
    for a, b in [(s[0], s[1]), (s[1], s[2]), (s[0], s[2])]:
        G.add_edge(int(a), int(b),
                   weight=float(np.hypot(*(pts[a]-pts[b]))) / 1000)

print(f"network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"connected? {nx.is_connected(G)}")
print(f"mean degree: {2*G.number_of_edges()/G.number_of_nodes():.2f}")

# the shortest path
src = names_d.index("Dhaka"); dst = names_d.index("Cox's Bazar")
path = nx.shortest_path(G, src, dst, weight="weight")
dist_km = nx.shortest_path_length(G, src, dst, weight="weight")
print(f"\nDhaka -> Cox's Bazar")
print(f"  route    : {' -> '.join(names_d[i] for i in path)}")
print(f"  steps    : {len(path)-1}")
print(f"  distance : {dist_km:,.0f} km (straight line, centroid to centroid)")

# centrality — which district is the most "central"?
btw = nx.betweenness_centrality(G, weight="weight")
top = sorted(btw.items(), key=lambda kv: -kv[1])[:5]
print("\nHighest betweenness centrality (the bridges of the network):")
for i, v in top:
    print(f"  {names_d[i]:16s} {v:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.4))

# (a) the network + the shortest path
dz_ok.plot(ax=ax[0], color="#f4f7fa", edgecolor="lightgray", linewidth=0.3)
cll = gpd.GeoSeries(cent, crs=3106).to_crs(4326)
for u, v_ in G.edges():
    ax[0].plot([cll.x.iloc[u], cll.x.iloc[v_]], [cll.y.iloc[u], cll.y.iloc[v_]],
               "-", color="lightsteelblue", lw=0.6, zorder=2)
px_ = [cll.x.iloc[i] for i in path]; py_ = [cll.y.iloc[i] for i in path]
ax[0].plot(px_, py_, "-o", color=C_WARM, lw=2.6, ms=6, zorder=5,
           label=f"Dhaka → Cox's Bazar ({dist_km:,.0f} km)")
ax[0].scatter(cll.x, cll.y, s=14, c=C_INK, zorder=4)
ax[0].set_title("(a) District connectivity network & shortest path")
ax[0].legend(fontsize=8.5, loc="upper left")

# (b) betweenness centrality
dz_ok["btw"] = [btw[i] for i in range(len(dz_ok))]
dz_ok.plot(column="btw", ax=ax[1], cmap="plasma", legend=True,
           edgecolor="white", linewidth=0.3,
           legend_kwds={"label": "Betweenness centrality", "shrink": 0.75})
ax[1].set_title("(b) Which districts act as connectors?")

for a in ax:
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

---
# Part 9 — Interactive maps (folium)

Static figures are for the journal. But for **exploration** and **presentation**
an interactive map is far more effective — zoom, click, switch layers.

> **⚠ File size:** passing 64 detailed polygons straight in makes the HTML several MB
> and the notebook heavy. So we lighten the geometry first with **`.simplify()`**.

In [ ]:
import folium

# simplifying the geometry — the size drops a lot, the look barely changes
simple = dz_ok[["ADM2_EN", "ADM1_EN", "trend", "t_mean",
                "rh_mean", "lisa", "zone_name", "geometry"]].copy()
simple["geometry"] = simple.geometry.simplify(0.005, preserve_topology=True)
print(f"GeoJSON before simplification ~ {len(dz_ok.to_json())/1e6:.2f} MB")
print(f"GeoJSON after simplification  ~ {len(simple.to_json())/1e6:.2f} MB")

m = folium.Map(location=[23.8, 90.4], zoom_start=7, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=simple.to_json(), data=simple,
    columns=["ADM2_EN", "trend"], key_on="feature.properties.ADM2_EN",
    fill_color="YlOrRd", fill_opacity=0.78, line_opacity=0.35,
    legend_name="Warming trend (°C per decade), 2000–2024",
    name="Warming trend",
).add_to(m)

folium.GeoJson(
    simple, name="District info",
    style_function=lambda x: {"fillOpacity": 0, "color": "#444", "weight": 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["ADM2_EN", "ADM1_EN", "trend", "t_mean", "rh_mean", "lisa"],
        aliases=["District:", "Division:", "Trend (°C/dec):",
                 "Mean T (°C):", "Mean RH (%):", "LISA class:"],
        localize=True, sticky=False),
).add_to(m)

folium.LayerControl().add_to(m)
m.save(OUT/"interactive_map.html")
print(f"\nSaved: outputs/interactive_map.html "
      f"({(OUT/'interactive_map.html').stat().st_size/1e6:.2f} MB)")
print("-> open it in a browser; hovering over a district shows its information.")
m

---
# Part 10 — Journal-ready figures and tables

## 10.1 What journals usually demand

| Requirement | Standard | What we are doing |
|------|-----|---------------|
| Resolution | >= 300 dpi (line art >= 600) | `dpi=300` |
| Format | TIFF / EPS / PDF (vector is better) | we save all three |
| Width | 1 column ~ 89 mm, 2 columns ~ 183 mm | `figsize` computed accordingly |
| Font | 7–9 pt, one family | fixed in rcParams |
| Colour | colour-blind friendly, readable in greyscale | ColorBrewer schemes |
| Panels | (a), (b), (c) labels | in every title |
| Scale bar and north arrow | mandatory on a map | added below |

> **Millimetres to inches:** 183 mm = 7.2 in (two columns),
> 89 mm = 3.5 in (one column).

In [ ]:
# small helper functions for drawing a scale bar and a north arrow
def add_north_arrow(ax, x=0.93, y=0.93, size=0.06):
    ax.annotate("N", xy=(x, y - size), xytext=(x, y),
                xycoords="axes fraction", textcoords="axes fraction",
                ha="center", va="center", fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6))

def add_scalebar(ax, gdf_ll, km=100, x=0.06, y=0.06):
    """An approximate scale bar — converts degrees to km at the central latitude."""
    lat_mid = float(gdf_ll.total_bounds[[1, 3]].mean())
    deg = km / (111.32 * np.cos(np.deg2rad(lat_mid)))
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    xs = x0 + (x1 - x0) * x; ys = y0 + (y1 - y0) * y
    ax.plot([xs, xs + deg], [ys, ys], "-", color="k", lw=3, solid_capstyle="butt")
    ax.text(xs + deg/2, ys + (y1-y0)*0.015, f"{km} km",
            ha="center", va="bottom", fontsize=8)

print("Helper functions ready")

In [ ]:
# =========== FIGURE 1 — the study area and the mean climate ===========
fig = plt.figure(figsize=(7.2, 6.4))                 # two-column width
gs = fig.add_gridspec(2, 2, hspace=0.26, wspace=0.28)

# (a) the study area
a = fig.add_subplot(gs[0, 0])
division.plot(ax=a, column="ADM1_EN", cmap="Pastel1", edgecolor="k", linewidth=0.4)
country.boundary.plot(ax=a, color="k", linewidth=1.1)
a.set_title("(a) Study area: 8 divisions", fontsize=9)
add_north_arrow(a); add_scalebar(a, division)

# (b) the ERA5 grid
b = fig.add_subplot(gs[0, 1])
t_mean_da.plot(ax=b, cmap="RdYlBu_r", add_colorbar=True,
               cbar_kwargs={"label": "Mean T (°C)", "shrink": 0.8})
country.boundary.plot(ax=b, color="k", linewidth=1.1)
b.set_title("(b) ERA5 grid, 0.25°", fontsize=9); b.set_xlabel(""); b.set_ylabel("")

# (c) temperature by district
c = fig.add_subplot(gs[1, 0])
dz_ok.plot(column="t_mean", ax=c, cmap="RdYlBu_r", legend=True,
           edgecolor="white", linewidth=0.2,
           legend_kwds={"label": "Mean T (°C)", "shrink": 0.8})
c.set_title("(c) District mean temperature", fontsize=9)

# (d) humidity by district
d = fig.add_subplot(gs[1, 1])
dz_ok.plot(column="rh_mean", ax=d, cmap="YlGnBu", legend=True,
           edgecolor="white", linewidth=0.2,
           legend_kwds={"label": "Mean RH (%)", "shrink": 0.8})
d.set_title("(d) District mean relative humidity", fontsize=9)

for a_ in [a, b, c, d]:
    a_.tick_params(labelsize=7)
    a_.set_xlabel("Longitude (°E)", fontsize=8)
    a_.set_ylabel("Latitude (°N)", fontsize=8)

fig.suptitle("Figure 1. Study area and mean climate, Bangladesh 2000–2024",
             fontsize=10, fontweight="bold", y=0.995)
plt.savefig(OUT/"Figure1_study_area.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT/"Figure1_study_area.pdf", bbox_inches="tight")
plt.show()
plt.close("all")
print("Figure 1 saved — png / pdf")

In [ ]:
# =========== FIGURE 2 — the warming trend ===========
fig = plt.figure(figsize=(7.2, 6.8))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3, height_ratios=[1, 1.05])

# (a) the national time series
a = fig.add_subplot(gs[0, :])
a.plot(years, t_ann.values, "o-", color=C_WARM, lw=1.3, ms=3.6, label="Annual mean")
a.plot(years, res_t.intercept + res_t.slope*years, "--", color=C_INK, lw=1.8,
       label=f"Trend {res_t.slope*10:+.3f} °C decade⁻¹ (p = {res_t.pvalue:.3f})")
# the +/-1 SE band — the line pivots about the data centroid (x-bar, y-bar), so it must be drawn that way
xbar, ybar = years.mean(), t_ann.values.mean()
a.fill_between(years,
               ybar + (res_t.slope - res_t.stderr) * (years - xbar),
               ybar + (res_t.slope + res_t.stderr) * (years - xbar),
               color=C_INK, alpha=0.18, label="±1 SE on slope")
pad = 0.35
a.set_ylim(t_ann.values.min() - pad, t_ann.values.max() + pad)
a.set_xlabel("Year", fontsize=8); a.set_ylabel("Temperature (°C)", fontsize=8)
a.set_title("(a) Area-weighted annual mean temperature", fontsize=9)
a.legend(fontsize=6.8, loc="upper left"); a.tick_params(labelsize=7)

# (b) the trend map + stippling
b = fig.add_subplot(gs[1, 0])
im = trend_bd.plot(ax=b, cmap="YlOrRd", vmin=0, vmax=0.45, add_colorbar=False)
cb = plt.colorbar(im, ax=b, shrink=0.82, pad=0.02)
cb.set_label("°C decade⁻¹", fontsize=8); cb.ax.tick_params(labelsize=7)
b.scatter(LO[sig_mask], LA[sig_mask], s=1.6, c="k", alpha=0.7, marker=".", linewidths=0)
country.boundary.plot(ax=b, color="k", linewidth=1.0)
b.set_title("(b) Trend (dots: p < 0.05)", fontsize=9)

# (c) the trend by district
c = fig.add_subplot(gs[1, 1])
dz_ok.plot(column="trend", ax=c, cmap="YlOrRd", legend=True,
           edgecolor="white", linewidth=0.2, vmin=0, vmax=0.45,
           legend_kwds={"label": "°C decade⁻¹", "shrink": 0.82})
division.boundary.plot(ax=c, color="k", linewidth=0.4)
c.set_title("(c) District-level trend", fontsize=9)

for a_ in [b, c]:
    a_.tick_params(labelsize=7)
    a_.set_xlabel("Longitude (°E)", fontsize=8); a_.set_ylabel("Latitude (°N)", fontsize=8)

fig.suptitle("Figure 2. Warming trends over Bangladesh, 2000–2024 (ERA5)",
             fontsize=10, fontweight="bold", y=0.995)
plt.savefig(OUT/"Figure2_trends.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT/"Figure2_trends.pdf", bbox_inches="tight")
plt.show()
plt.close("all")
print("Figure 2 saved — png / pdf")

In [ ]:
# =========== FIGURE 3 — the spatial structure ===========
fig = plt.figure(figsize=(7.2, 6.6))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# (a) Moran scatterplot
a = fig.add_subplot(gs[0, 0])
a.scatter(z_std, z_lag, s=16, color=C_COOL, edgecolor="k", linewidth=0.3, alpha=0.8)
a.plot(mor_xs, mor_a + mor_b*mor_xs, "-", color=C_WARM, lw=1.6)
a.axhline(0, color="k", lw=0.6); a.axvline(0, color="k", lw=0.6)
a.set_xlabel("Standardised trend (z)", fontsize=8)
a.set_ylabel("Spatial lag", fontsize=8)
a.set_title(f"(a) Moran's I = {mi.I:.3f} (p = {mi.p_sim:.3f})", fontsize=9)

# (b) LISA
bx = fig.add_subplot(gs[0, 1])
dz_ok.plot(ax=bx, color=dz_ok["lisa"].map(cmap_d), edgecolor="white", linewidth=0.25)
bx.set_title("(b) LISA clusters", fontsize=9)
bx.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in cmap_d.items()],
          loc="upper right", fontsize=5.4, frameon=True, framealpha=0.92,
          borderpad=0.35, handlelength=1.1)

# (c) Gi*
cx = fig.add_subplot(gs[1, 0])
dz_ok.plot(column="gi_z", ax=cx, cmap="RdBu_r", vmin=-3, vmax=3, legend=True,
           edgecolor="white", linewidth=0.25,
           legend_kwds={"label": "Gi* z-score", "shrink": 0.82})
cx.set_title("(c) Getis-Ord Gi* hot-spots", fontsize=9)

# (d) the clusters
dx = fig.add_subplot(gs[1, 1])
dz_ok.plot(ax=dx, color=dz_ok["zone_name"].map(zone_cols),
           edgecolor="white", linewidth=0.25)
dx.set_title(f"(d) Climate zones (k = {K})", fontsize=9)
dx.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in zone_cols.items()],
          loc="upper right", fontsize=5.4, frameon=True, framealpha=0.92,
          borderpad=0.35, handlelength=1.1)

for a_ in [bx, cx, dx]:
    division.boundary.plot(ax=a_, color="k", linewidth=0.35)
    a_.set_xlabel("Longitude (°E)", fontsize=8); a_.set_ylabel("Latitude (°N)", fontsize=8)
for a_ in [a, bx, cx, dx]:
    a_.tick_params(labelsize=7)

fig.suptitle("Figure 3. Spatial structure of warming across Bangladesh",
             fontsize=10, fontweight="bold", y=0.995)
plt.savefig(OUT/"Figure3_spatial_structure.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT/"Figure3_spatial_structure.pdf", bbox_inches="tight")
plt.show()
plt.close("all")
print("Figure 3 saved — png / pdf")

In [ ]:
# =========== Tables ===========
# Table 1 — the summary by division
tbl1 = (dz_ok.groupby("ADM1_EN")
        .agg(Districts=("ADM2_EN", "size"),
             Mean_T_C=("t_mean", "mean"), Mean_RH_pct=("rh_mean", "mean"),
             Trend_C_dec=("trend", "mean"), Sen_C_dec=("sen", "mean"))
        .round(3).sort_values("Trend_C_dec", ascending=False))
tbl1.to_csv(OUT/"Table1_division_summary.csv")

# Table 2 — the full results by district
tbl2 = (dz_ok[["ADM2_EN", "ADM1_EN", "t_mean", "rh_mean", "trend", "sen",
               "pval", "lisa", "gi_z", "zone_name"]]
        .rename(columns={"ADM2_EN": "District", "ADM1_EN": "Division",
                         "t_mean": "Mean_T_C", "rh_mean": "Mean_RH_pct",
                         "trend": "OLS_trend_C_dec", "sen": "Sen_slope_C_dec",
                         "pval": "p_value", "lisa": "LISA_class",
                         "gi_z": "GiStar_z", "zone_name": "Climate_zone"})
        .sort_values("OLS_trend_C_dec", ascending=False).round(4))
tbl2.to_csv(OUT/"Table2_district_results.csv", index=False)

# Table 3 — the summary of methods (for the Methods section)
tbl3 = pd.DataFrame([
    ["Data source", "ERA5 reanalysis, ECMWF (Copernicus CDS)"],
    ["Variables", "Air temperature (t), relative humidity (r) at 1000 hPa"],
    ["Period", f"{int(years[0])}–{int(years[-1])} ({len(years)} years, {ds.sizes['valid_time']} months)"],
    ["Spatial resolution", "0.25° × 0.25° (~28 km)"],
    ["Domain", "20–27°N, 87–94°E"],
    ["Boundaries", "BBS/HDX administrative boundaries, 2020-11-13"],
    ["Zonal statistics", "all_touched rasterisation (rioxarray)"],
    ["Spatial averaging", "cos(latitude) area weighting"],
    ["Trend method", "OLS linear regression + Mann-Kendall + Theil-Sen"],
    ["Significance", "p < 0.05, two-tailed"],
    ["Spatial autocorrelation", "Global Moran's I, LISA, Getis-Ord Gi* (999 permutations)"],
    ["Spatial weights", f"{'Queen contiguity' if isinstance(wq, Queen) else 'KNN (k=5)'}, row-standardised"],
    ["Clustering", f"K-Means (k={K}), standardised features"],
    ["Software", f"Python, geopandas {gpd.__version__}, xarray {xr.__version__}, esda"],
], columns=["Item", "Specification"])
tbl3.to_csv(OUT/"Table3_methods.csv", index=False)

print("═══ Table 1 — Division summary ═══")
print(tbl1.to_string())
print("\n═══ Table 2 — top 10 districts by warming ═══")
print(tbl2.head(10).to_string(index=False))
print("\nTables saved: Table1, Table2, Table3 (CSV)")

In [ ]:
# =========== Results summary (a draft of the Abstract/Results) ===========
n_sig = int((dz_ok["pval"] < 0.05).sum())
hh_list = sorted(dz_ok[dz_ok["lisa"] == "HH (hot-spot)"]["ADM2_EN"].tolist())
ll_list = sorted(dz_ok[dz_ok["lisa"] == "LL (cold-spot)"]["ADM2_EN"].tolist())

summary = f"""
RESULTS SUMMARY  (draft text for a manuscript)
{'='*66}

Area-weighted mean near-surface air temperature over Bangladesh rose at
{res_t.slope*10:+.3f} °C per decade during {int(years[0])}–{int(years[-1])}
(OLS, p = {res_t.pvalue:.3f}; Mann-Kendall {mk_t.trend}, p = {mk_t.p:.3f};
Theil-Sen {mk_t.slope*10:+.3f} °C per decade), equivalent to a total warming
of {res_t.slope*len(years):+.2f} °C over the {len(years)}-year period.

Relative humidity showed no significant change
({res_r.slope*10:+.3f} % per decade, p = {res_r.pvalue:.3f}).

Warming was spatially non-uniform, ranging from {dz_ok['trend'].min():+.3f} to
{dz_ok['trend'].max():+.3f} °C per decade across the {len(dz_ok)} districts, and was
statistically significant (p < 0.05) in {n_sig} districts ({100*n_sig/len(dz_ok):.0f} %).

The warming field was strongly spatially clustered
(global Moran's I = {mi.I:.3f}, p = {mi.p_sim:.3f}). LISA analysis identified
{len(hh_list)} high-high clusters ({', '.join(hh_list[:5])}{'...' if len(hh_list)>5 else ''})
and {len(ll_list)} low-low clusters ({', '.join(ll_list[:5])}{'...' if len(ll_list)>5 else ''}).

K-means clustering of mean temperature, humidity and warming rate yielded
{K} coherent climate zones.
{'='*66}
"""
print(summary)
with open(OUT/"results_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary)

# reproducibility information
import sys, platform
meta = {
    "generated": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "python": sys.version.split()[0], "platform": platform.system(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "geopandas": gpd.__version__, "xarray": xr.__version__,
}
pd.Series(meta).to_csv(OUT/"reproducibility.csv", header=["value"])
print("Environment information saved: reproducibility.csv")

print("\n=== Every file created in the outputs/ folder ===")
for f in sorted(OUT.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s} {f.stat().st_size/1e6:8.2f} MB")

---
# Home Work — Class 11

**1.** Run the whole trend analysis for relative humidity (`ds.r`) —
per-pixel OLS, clipped to Bangladesh, with a stippled map. Is the humidity
trend significant anywhere?

**2.** Compute Moran's I with `KNN` (k = 4, 6, 8) instead of `Queen`.
How much does I change when the definition of a neighbour changes? What does that tell you?

**3.** Run K-Means with `K = 3` and `K = 5`. Which is best by the silhouette
score? Are the zones geographically meaningful?

**4.** Compute the trend for the **monsoon (JJAS)** months only and compare it with the
annual trend. Is the monsoon warming faster than the year as a whole?

**5.** Build a **Figure 4** — of your own choosing, but it must be two columns wide
(7.2 in), at least 300 dpi, with (a)(b) panel labels, a scale bar and a north arrow.
Save it in both TIFF and PDF.

> **A challenge:** open `Table2_district_results.csv` — what is the warming rate for
> your own district? Is it significant? Which LISA class did it fall into?

---
# Class 11 — Summary

| What we learned | Key command |
|-----------|-------------|
| area-weighted mean | `.weighted(np.cos(np.deg2rad(lat))).mean()` |
| linear trend | `stats.linregress(x, y)` |
| fast per-pixel trend | vectorized `Sxy / Sxx` |
| non-parametric trend | `mk.original_test(series)` |
| spatial weights | `Queen.from_dataframe()` · `KNN.from_dataframe()` |
| global autocorrelation | `esda.Moran(y, w)` |
| local clusters | `esda.Moran_Local(y, w)` |
| hot-spots | `esda.G_Local(y, w)` |
| clustering | `KMeans(n_clusters=k).fit(StandardScaler().fit_transform(X))` |
| networks | `nx.shortest_path(G, s, t, weight="weight")` |
| interactive maps | `folium.Choropleth()` |
| publication figures | `plt.savefig(..., dpi=300, bbox_inches="tight")` |

---

### The main findings (the summary of your study)

- Bangladesh warmed by **about +0.2 °C per decade** between 2000 and 2024, statistically significantly
- Relative humidity shows **no significant change**
- The warming is **spatially uneven**, and **strongly clustered** (Moran's I ~ 0.8)
- **Hot-spots** in the eastern hill districts, **cold-spots** along the south-western coast

### The most common mistakes

1. **Not weighting by area** — in a grid mean the northern cells get too much weight.
2. **Not showing significance** — a trend map alone is not enough; add stippling.
3. **Not standardising before K-Means** — the largest-scale feature takes over the result.
4. **Ignoring islands in Moran's I** — if there are disconnected units, use KNN.
5. **Not documenting the methods** — write down the CRS, the boundary version, the zonal method.
6. **Saving figures at 72 dpi** — the journal will send them back; always use at least 300 dpi.

### Course complete — what you can now do

Across the four notebooks you can now carry out a complete geospatial study:
**read the data -> fix the CRS -> match vector to raster -> statistics ->
spatial analysis -> publication figures and tables**.

That framework applies to any dataset — rainfall, floods, land cover,
population, agriculture — just change the input, the method stays the same.

---

**Md Wabidur Rahman** — Researcher, Research and Integrated Thought (RIT)
*Geospatial Data Analysis and Visualization · Class 11 — Advanced Analysis & Geovisualization*